[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Class and Static Methods


## What you will be able to do

Write methods that belong to a class rather than to one object: alternative constructors such as
`Station.from_csv(line)`, which build an object from a line of a file or a JSON record, static
methods for calculations that need no object, and attributes shared by every object of a class.


## The idea

### The problem

Every method written so far takes `self`, because it works on one object that already exists. Two
kinds of work do not fit that shape.

The first is building an object. A station arrives as a line of a CSV file, `"Tromso,-4.1,-2.6"`,
or as a record from a JSON file, and `__init__` takes neither: it takes a name and a list of
numbers. The parsing has to live somewhere. In the calling code, every place that reads a station
repeats it. In `__init__`, with an optional argument for each format, the constructor becomes a
switchboard: which arguments go together, which one wins when two are passed, and what happens when
none are, all become questions a reader has to answer by reading the branches. And it cannot be an
ordinary method, because an ordinary method needs a station to be called on, and the whole point is
that there is not one yet.

The second is a calculation that belongs with the class but needs no object, such as converting a
Celsius reading to Fahrenheit. As an ordinary method it takes a `self` it never uses, and it cannot
be called until some station exists.

### What class and static methods are

> A **class method**, marked `@classmethod`, receives the class itself as its first argument,
> conventionally named `cls`, instead of an object. It is called on the class, as
> `Station.from_csv(line)`, before any station exists, and its usual job is to build one and return
> it. A **static method**, marked `@staticmethod`, receives neither: it is a plain function kept
> inside a class because that is where a reader will look for it.

### Why it works that way

The **Decorators** notebook listed both in its table: `@classmethod` passes the class instead of an
object, and `@staticmethod` passes neither. Everything else follows from that one difference in the
first argument.

A class method's main use is the alternative constructor. `__init__` takes clean values, and one
class method per input format turns a messy input into those values and calls the constructor. Each
has a name that says what it takes, such as `from_csv` or `from_dict`, so a caller never has to work
out what a combination of optional arguments means, and the parsing lives inside the class it
belongs to.

The method builds with `cls(...)` rather than naming `Station`. `cls` is whichever class the method
was called on, and the **Inheritance** notebook is where that starts to matter.

A class can also hold data of its own, assigned in the class body rather than in `__init__`, such
as the units a station accepts. Every object reads the same value. That sharing is useful for
constants and dangerous for anything that changes, which is the quiet error at the end.

### Where you will meet this

The standard library is full of alternative constructors. `datetime.fromisoformat("2026-03-01")`
builds a date from a string, `dict.fromkeys` builds a dictionary from a list of keys, and
`Path.cwd()` from the **Paths** notebook builds a path without needing one first. In the **Pandas**
guide, `DataFrame.from_dict` and `DataFrame.from_records` follow exactly this pattern.

### What this notebook covers

One constructor with an option per format against one class method per format, then what `cls` is,
class attributes, static methods and when a plain function is better, the standard library's own
constructors, and one class that uses all of it.

### A first look

An alternative constructor. There is nothing to run yet: read it, and read the output underneath it.

```python
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])


north = Station.from_csv("Tromso,-4.1,-2.6")
print(north.name, north.readings)
```

```
Tromso [-4.1, -2.6]
```

`from_csv` was called on `Station` itself, because no station existed yet. It built one and handed it
back.


## Setup

Three imports.

- `json` turns a JSON string into the dictionary that one of the constructors builds a station from
- `datetime` supplies one of the standard library's own alternative constructors, in one section
  near the end
- `Path` supplies another, `Path.cwd()`, in the same section

Every class in this notebook is written in the section that uses it.

**Run this cell before the rest of the notebook.**


In [1]:
import json
from datetime import datetime
from pathlib import Path

print("ready")


ready


## Worked examples

### Before and after: one constructor, or one per format

Here is the problem from the top of this notebook, in code. A station can arrive three ways: as a
name and a list of numbers, as a line of a CSV file, or as a record from a JSON file. Each version of
the class is put through the same five steps, numbered in the code and in the output:

1. Build a station from a name and readings.
2. Build one from a CSV line.
3. Build one from a JSON record.
4. A mistake: pass a name and a CSV line together. It should be refused.
5. A mistake: pass nothing at all. It should be refused.

First, the design most people reach for: one `__init__` with an optional argument for each format,
and a branch for each.


In [2]:
class OneInit:
    def __init__(self, name=None, readings=None, csv_line=None, record=None):
        if csv_line is not None:
            fields = csv_line.split(",")
            name, readings = fields[0], [float(v) for v in fields[1:]]
        elif record is not None:
            name, readings = record["name"], record["readings"]
        self.name = name
        self.readings = readings


The five steps.


In [3]:
# 1. Build a station from a name and readings.
a = OneInit("Tromso", [-4.1, -2.6])
print("1.", a.name, a.readings)

# 2. Build one from a CSV line.
b = OneInit(csv_line="Bodo,-2.6,-1.9")
print("2.", b.name, b.readings)

# 3. Build one from a JSON record.
c = OneInit(record=json.loads('{"name": "Malaga", "readings": [18.9, 19.4]}'))
print("3.", c.name, c.readings)

# 4. A mistake: a name and a CSV line together. It should be refused.
d = OneInit("Galway", csv_line="Bodo,-2.6,-1.9")
print("4.", d.name, d.readings)

# 5. A mistake: nothing at all. It should be refused.
e = OneInit()
print("5.", e.name, e.readings)


1. Tromso [-4.1, -2.6]
2. Bodo [-2.6, -1.9]
3. Malaga [18.9, 19.4]
4. Bodo [-2.6, -1.9]
5. None None


Steps 1 to 3 work. Steps 4 and 5 are the problem, and neither raised an error.

Step 4 built a station called Bodo. The CSV branch ran and replaced the name `'Galway'` without a
word, so a station exists with a name its caller never gave it. Step 5 built a station whose name and
readings are both `None`, which will fail much later, on some line that has nothing to do with this
one. The branches decide what each combination means, and the only way for a caller to find out is
to read them.

Now the same class with one plain `__init__`, and one class method for each other format.


In [4]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])

    @classmethod
    def from_dict(cls, record):
        return cls(record["name"], record["readings"])


The same five steps. Steps 2 and 3 now call a named method on the class, instead of passing a
keyword to `__init__`. `name, *values` is the starred assignment from the **Tuples and Unpacking**
notebook: the first field goes to `name`, and the rest go to `values` as a list.


In [5]:
# 1. Build a station from a name and readings.
a = Station("Tromso", [-4.1, -2.6])
print("1.", a.name, a.readings)

# 2. Build one from a CSV line.
b = Station.from_csv("Bodo,-2.6,-1.9")
print("2.", b.name, b.readings)

# 3. Build one from a JSON record.
c = Station.from_dict(json.loads('{"name": "Malaga", "readings": [18.9, 19.4]}'))
print("3.", c.name, c.readings)

# 4. A mistake: a name and a CSV line together. It should be refused.
try:
    d = Station.from_csv("Bodo,-2.6,-1.9", "Galway")
    print("4.", d.name, d.readings)
except TypeError as error:
    print("4. refused:", error)

# 5. A mistake: nothing at all. It should be refused.
try:
    e = Station()
    print("5.", e.name, e.readings)
except TypeError as error:
    print("5. refused:", error)


1. Tromso [-4.1, -2.6]
2. Bodo [-2.6, -1.9]
3. Malaga [18.9, 19.4]
4. refused: Station.from_csv() takes 2 positional arguments but 3 were given
5. refused: Station.__init__() missing 2 required positional arguments: 'name' and 'readings'


Steps 1 to 3 built the same three stations as before. Steps 4 and 5 were refused, on the lines that
made the mistakes, with messages that name the problem: `from_csv` takes one line, and `__init__`
needs a name and readings. Each way of building a station is now its own door, with its own name,
and no combination of arguments is left to get wrong.

| | One `__init__` with options | One class method per format |
|---|---|---|
| Building from a CSV line | `OneInit(csv_line=line)` | `Station.from_csv(line)` |
| Building from a JSON record | `OneInit(record=record)` | `Station.from_dict(record)` |
| Step 4, a name and a line together | builds Bodo, and the name is lost | refused with `TypeError` |
| Step 5, no arguments | builds a station of `None` values | refused with `TypeError` |
| Adding another format | another option and another branch in `__init__` | another class method, and `__init__` untouched |

The rest of this notebook takes the class-method version apart.

| Question about the class-method version | The section that answers it |
|---|---|
| What is `cls`, and who passes it? | What `cls` is |
| Why `cls(...)` and not `Station(...)`? | What `cls` is |
| Where would a list of allowed units go? | Data that belongs to the class |
| Where does a conversion that needs no station go? | Static methods |
| Why not just write a plain function? | A class method, a static method, or a function |

### What `cls` is

`@classmethod` changes what the first argument is. For an ordinary method Python passes the object,
and for a class method it passes the class. This one returns whatever it receives.


In [6]:
class Probe:
    @classmethod
    def which(cls):
        return cls


print("Probe.which() returned:", Probe.which())
print("it is Probe itself:    ", Probe.which() is Probe)

probe = Probe()
print("called through an object, cls is still the class:", probe.which() is Probe)

print()
print("Station.from_csv is a", type(Station.from_csv).__name__,
      "bound to", Station.from_csv.__self__.__name__)


Probe.which() returned: <class '__main__.Probe'>
it is Probe itself:     True
called through an object, cls is still the class: True

Station.from_csv is a method bound to Station


`cls` is the class itself, not an object made from it. Calling the method through an object changes
nothing: Python passes the object's class.

So `cls(...)` builds a new object exactly as `Probe(...)` would, and in `from_csv`, `cls(name, ...)`
runs `Station.__init__`. The last line connects this to the **Methods** notebook: reached on the class,
`from_csv` is already a bound method, bound to the class in the way `north.average` was bound to
`north`.

### Data that belongs to the class

An attribute assigned in the class body, outside every method, belongs to the class. Every object
reads it, and no object holds a copy of its own. A tuple of allowed units is a natural one, and
`__init__` reads it through `self`.


In [7]:
class Station:
    UNITS = ("C", "F")

    def __init__(self, name, unit="C"):
        if unit not in self.UNITS:
            raise ValueError(f"unit must be one of {self.UNITS}, not {unit!r}")
        self.name = name
        self.unit = unit


north = Station("Tromso")
south = Station("Malaga", "F")

print("Station.UNITS:", Station.UNITS)
print("north.UNITS:  ", north.UNITS)
print("one shared object:", north.UNITS is south.UNITS is Station.UNITS)
print("what north itself holds:", vars(north))


Station.UNITS: ('C', 'F')
north.UNITS:   ('C', 'F')
one shared object: True
what north itself holds: {'name': 'Tromso', 'unit': 'C'}


`vars(north)` holds only `name` and `unit`. Reading `north.UNITS` found nothing on the object, so
Python looked on the class and found it there. Every station shares the one tuple.

Assigning through an object works differently from reading through it.


In [8]:
north.UNITS = ("K",)

print("north.UNITS:  ", north.UNITS)
print("south.UNITS:  ", south.UNITS)
print("Station.UNITS:", Station.UNITS)
print("now in what north holds:", "UNITS" in vars(north))

del north.UNITS
print("after del north.UNITS:", north.UNITS)


north.UNITS:   ('K',)
south.UNITS:   ('C', 'F')
Station.UNITS: ('C', 'F')
now in what north holds: True
after del north.UNITS: ('C', 'F')


The assignment did not change the class attribute. It created a new attribute on `north` alone,
which hid the class's value from `north` and from nobody else. Deleting it let the class's value show
through again. This is the same mechanism as the stray attribute in the **Properties** notebook: an
assignment through an object always lands on the object.

### Static methods

A method that never touches `self` still has to be called on an object, which is the wrong shape for a
conversion that has nothing to do with any particular station.


In [9]:
class Habit:
    def to_fahrenheit(self, celsius):
        return round(celsius * 9 / 5 + 32, 1)


print("on an object:", Habit().to_fahrenheit(-4.1))

try:
    Habit.to_fahrenheit(-4.1)
except TypeError as error:
    print("on the class:", error)


on an object: 24.6
on the class: Habit.to_fahrenheit() missing 1 required positional argument: 'celsius'


Called on the class, the reading went into `self`, and `celsius` was left empty. `@staticmethod`
tells Python to pass nothing extra.


In [10]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @staticmethod
    def to_fahrenheit(celsius):
        return round(celsius * 9 / 5 + 32, 1)

    def in_fahrenheit(self):
        return [self.to_fahrenheit(c) for c in self.readings]


north = Station("Tromso", [-4.1, -2.6])

print("on the class:     ", Station.to_fahrenheit(-4.1))
print("on an object:     ", north.to_fahrenheit(-4.1))
print("used by a method: ", north.in_fahrenheit())
print("reached on the class, it is a plain", type(Station.to_fahrenheit).__name__)


on the class:      24.6
on an object:      24.6
used by a method:  [24.6, 27.3]
reached on the class, it is a plain function


The same function works on the class or on an object, and the class's own methods reach it through
`self`. Reached on the class it is a plain `function`, because `@staticmethod` binds nothing.

### A class method, a static method, or a function

| The method needs | Write | Called as |
|---|---|---|
| the object's data | an ordinary method, taking `self` | `north.in_fahrenheit()` |
| the class, usually to build an object | `@classmethod`, taking `cls` | `Station.from_csv(line)` |
| neither | `@staticmethod`, or a plain function in the module | `Station.to_fahrenheit(-4.1)` |

The last row has two answers, and a plain function is usually fine. Here is the same conversion as
one.


In [11]:
def to_fahrenheit(celsius):
    return round(celsius * 9 / 5 + 32, 1)


print(to_fahrenheit(-4.1))


24.6


It does the same job with nothing to explain. Choose the static method when the function only makes
sense for this class, so callers should find it on the class, next to the readings it converts.
Choose the plain function when code with no stations in it would use it too. The **Why Classes**
notebook made the same argument about whole classes.

### The standard library's own

Alternative constructors are the usual way for the standard library to offer a second way of building
something.


In [12]:
print("datetime.fromisoformat:", repr(datetime.fromisoformat("2026-03-01")))
print("dict.fromkeys:         ", dict.fromkeys(["Tromso", "Bodo"], 0))
print("int.from_bytes:        ", int.from_bytes(b"\x01\x00", "big"))
print("Path.cwd() is a Path:  ", isinstance(Path.cwd(), Path))


datetime.fromisoformat: datetime.datetime(2026, 3, 1, 0, 0)
dict.fromkeys:          {'Tromso': 0, 'Bodo': 0}
int.from_bytes:         256
Path.cwd() is a Path:   True


Each is called on a class rather than on an existing object, and each hands back a new one: a date
from a string, a dictionary from a list of keys, a number from two bytes, and a path. `Path.cwd()` is
not printed itself because it is the folder this notebook runs in, which is different on every
machine.

### Putting it together: stations from a file and from JSON

One class using everything above. A class attribute lists the allowed units. Two alternative
constructors read a CSV line and a JSON record. A third class method builds many stations at once by
calling one of the others through `cls`. A static method converts, and an ordinary method uses it.

The CSV lines now carry a unit after the name, and one of them is blank, as real files often are.


In [13]:
class Station:
    """A weather station that can be built from a CSV line or a JSON record."""

    UNITS = ("C", "F")

    def __init__(self, name, readings, unit="C"):
        if unit not in self.UNITS:
            raise ValueError(f"unit must be one of {self.UNITS}, not {unit!r}")
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"

    @classmethod
    def from_csv(cls, line):
        name, unit, *values = line.strip().split(",")
        return cls(name, [float(v) for v in values], unit)

    @classmethod
    def from_dict(cls, record):
        return cls(record["name"], record["readings"], record.get("unit", "C"))

    @classmethod
    def load(cls, lines):
        return [cls.from_csv(line) for line in lines if line.strip()]

    @staticmethod
    def to_fahrenheit(celsius):
        return round(celsius * 9 / 5 + 32, 1)

    def in_fahrenheit(self):
        if self.unit == "F":
            return list(self.readings)
        return [self.to_fahrenheit(c) for c in self.readings]


archive = ["Tromso,C,-4.1,-2.6", "", "Malaga,F,66.0,67.1", "Bodo,C,-2.6,-1.9"]

for station in Station.load(archive):
    print(station, "->", station.in_fahrenheit())


Station('Tromso', [-4.1, -2.6], 'C') -> [24.6, 27.3]
Station('Malaga', [66.0, 67.1], 'F') -> [66.0, 67.1]
Station('Bodo', [-2.6, -1.9], 'C') -> [27.3, 28.6]


`load` skipped the blank line and handed every other line to `from_csv` through `cls`, so there is
one piece of parsing code however many lines arrive. Each station knows its own unit, and
`in_fahrenheit` converts only the ones recorded in Celsius.

Now a JSON record, and a line with a unit the class does not accept.


In [14]:
galway = Station.from_dict(json.loads('{"name": "Galway", "readings": [11.5, 12.1]}'))
print(galway, "with the unit defaulted to", repr(galway.unit))

try:
    Station.from_csv("Oslo,K,270.1")
except ValueError as error:
    print("refused:", error)

print("the conversion on its own:", Station.to_fahrenheit(0.0))


Station('Galway', [11.5, 12.1], 'C') with the unit defaulted to 'C'
refused: unit must be one of ('C', 'F'), not 'K'
the conversion on its own: 32.0


The JSON record had no unit, so `from_dict` supplied `'C'`. The Oslo line reached `__init__` through
`from_csv` and was refused there, because every constructor ends in `__init__` and `__init__` checks
the unit against `UNITS`. The rule is written once and every door passes through it.

### Where each part came from

| In `Station` | What it relies on | The section that showed it |
|---|---|---|
| `from_csv` and `from_dict` | one named class method per input format | Before and after |
| `return cls(...)` | `cls` is the class the method was called on | What `cls` is |
| `load` calling `cls.from_csv` | a class method can reach the others through `cls` | What `cls` is |
| `UNITS` | data that belongs to the class, shared by every object | Data that belongs to the class |
| `to_fahrenheit` | a function that needs no object | Static methods |
| `self.to_fahrenheit(c)` inside a method | a static method is reachable through `self` | Static methods |
| every constructor refusing `'K'` | each one ends in `__init__`, which checks | Before and after |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/08-class-and-static-methods-solutions.ipynb).

**1.** Give a `Book` class, storing `title`, `author` and `pages`, a class method `from_csv(line)`
for lines like `"Dune,Herbert,412"`, with `pages` converted to an integer. Build two books from two
lines and print each one's title and pages.


In [15]:
# your code here


**2.** Add a second alternative constructor, `from_dict(record)`, and use it on the dictionary that
`json.loads` returns for `'{"title": "Emma", "author": "Austen", "pages": 474}'`.


In [16]:
# your code here


**3.** Give `Book` a class attribute `count = 0` that `__init__` increases by one with
`Book.count += 1`, and a class method `how_many()` that returns it. Create three books, then print
`Book.how_many()`.


In [17]:
# your code here


**4.** Give `Book` a static method `pages_for(words)` that estimates a page count at 300 words a page,
as `words // 300 + 1`. Call it on the class and on a book.


In [18]:
# your code here


**5.** Change the counter so that `__init__` uses `self.count += 1` instead of `Book.count += 1`.
Create three books, then print `Book.count` and each book's `count`. Say in a comment what happened.


In [19]:
# your code here


**6.** Write a class method `load(lines)` that builds a list of books from a list of CSV lines,
skipping blank ones, and use it on three lines where the middle one is empty.


In [20]:
# your code here


## Common errors

### TypeError: `@classmethod` left off

Without the decorator, `from_csv` is an ordinary function stored in the class.


In [21]:
class NoDecorator:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])


NoDecorator.from_csv("Tromso,-4.1,-2.6")


TypeError: NoDecorator.from_csv() missing 1 required positional argument: 'line'

The message says `line` is missing, and a line was plainly passed. Called on the class, an undecorated
function receives exactly what the call supplies: the string went into the first parameter, `cls`, and
`line` was left empty. When the argument you can see yourself passing is reported missing, and it is
the last parameter, the decorator is usually the thing that is missing.

### NameError: `self` inside a class method

A class method runs before there is an object, so it has no `self`.


In [22]:
class SelfInside:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        self.name = name
        return cls(name, [float(v) for v in values])


SelfInside.from_csv("Tromso,-4.1,-2.6")


NameError: name 'self' is not defined

Its first parameter is `cls`, and `self` is simply not a name inside it. Anything a class method wants
set on the new object goes through `cls(...)`, which runs `__init__`, and `__init__` has the `self`.

### AttributeError: the constructor forgot to return

`cls(...)` builds the object. The method still has to hand it back.


In [23]:
class NoReturn:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        cls(name, [float(v) for v in values])


north = NoReturn.from_csv("Tromso,-4.1,-2.6")
print("north is:", north)
north.name


north is: None


AttributeError: 'NoneType' object has no attribute 'name'

The station was built and immediately thrown away, and the method returned `None`. The error then
arrives on the next line that uses the result, naming `NoneType` rather than anything about the
constructor. An alternative constructor ends with `return cls(...)`.

### The quiet one: a mutable class attribute

A class attribute is shared by every object, which is exactly right for a tuple of units. For a list
that each object is meant to fill with its own data, it is exactly wrong.


In [24]:
class Shared:
    readings = []

    def __init__(self, name):
        self.name = name

    def add(self, value):
        self.readings.append(value)


north = Shared("Tromso")
south = Shared("Malaga")
north.add(-4.1)

print("north.readings:", north.readings)
print("south.readings:", south.readings)
print("one list:      ", north.readings is south.readings)


north.readings: [-4.1]
south.readings: [-4.1]
one list:       True


A reading added to Tromso appeared at Malaga, and nothing raised. `self.readings.append` found no
`readings` on the object, went to the class, and changed the one list every station shares. Unlike the
tuple assignment earlier, `append` changes the list in place, so no object gets an attribute of its
own.

It is the class-level version of the mutable default in the **Your First Class** notebook, and the fix
is the same idea: build a new list for each object, in `__init__`.


In [25]:
class Separate:
    def __init__(self, name):
        self.name = name
        self.readings = []

    def add(self, value):
        self.readings.append(value)


north = Separate("Tromso")
south = Separate("Malaga")
north.add(-4.1)

print("north.readings:", north.readings)
print("south.readings:", south.readings)
print("one list:      ", north.readings is south.readings)


north.readings: [-4.1]
south.readings: []
one list:       False


Keep class attributes for values that never change, such as `UNITS`. Anything an object fills in
belongs in `__init__`.


## Recap

- `@classmethod` passes the class as `cls`, so the method can be called on the class before any
  object exists.
- An alternative constructor turns one input format into the values `__init__` takes, and returns
  `cls(...)`.
- One named class method per format is clearer than one `__init__` with an optional argument for each.
- `cls` is whichever class the method was called on, even when it is called through an object.
- A class method can call another through `cls`, as `load` calls `from_csv`.
- An attribute assigned in the class body belongs to the class, and every object reads the same one.
- Assigning through an object creates the object's own attribute, which hides the class's.
- A mutable class attribute is shared by every object. Give each object its own in `__init__`.
- `@staticmethod` passes nothing extra. It is a plain function kept with its class.
- A plain function in the module is often just as good. Choose by where a reader should find it.
- A missing argument you can see yourself passing, or `self` not defined, points at the decorator.


## What is next

The **Inheritance** notebook. `cls(...)` was chosen over `Station(...)` for a reason this notebook
could only promise: a class can be built on top of another, borrowing everything it has and changing
only what differs, and a class method inherited that way builds the right kind of object. That
notebook covers `class Child(Parent)`, `super()`, and the order in which Python looks for a method.


---

&#8592; **Previous:** [Properties](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/07-properties.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Inheritance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/09-inheritance.ipynb) &#8594;
